In [ ]:
# Setup
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

from data import load_data
from model import (build_tfidf_pipeline, split_data, evaluate_classifier,
                   train_distilbert, topic_model_negative_reviews,
                   save_pipeline, save_json, RANDOM_SEED)

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
ARTEFACTS_DIR.mkdir(exist_ok=True)
np.random.seed(RANDOM_SEED)

In [ ]:
# 1. Load + split data
df = load_data()
X_train, X_test, y_train, y_test = split_data(df)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Class distribution (train): {pd.Series(y_train).value_counts().to_dict()}")

In [ ]:
# 2. TF-IDF + Logistic Regression baseline
print("Training TF-IDF baseline...")
pipe = build_tfidf_pipeline()
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
tfidf_metrics = evaluate_classifier(y_test, y_pred)
print(f"  Macro F1 = {tfidf_metrics['f1_macro']:.4f}")
print(f"  Weighted F1 = {tfidf_metrics['f1_weighted']:.4f}")
save_pipeline(pipe, 'tfidf_pipeline', ARTEFACTS_DIR)

In [ ]:
# 3. (Optional) DistilBERT fine-tuning
print("Trying DistilBERT fine-tune (skips if torch/transformers not installed)...")
bert_pred, bert_metrics, bert_info = train_distilbert(X_train, y_train, X_test, y_test, epochs=2)
if bert_metrics is None:
    print(f"  DistilBERT skipped: {bert_info.get('error', 'unknown reason')}")
else:
    print(f"  Macro F1 = {bert_metrics['f1_macro']:.4f}")
    print(f"  Weighted F1 = {bert_metrics['f1_weighted']:.4f}")

In [ ]:
# 4. BERTopic on negative reviews
print("Running BERTopic on negative reviews...")
topics = topic_model_negative_reviews(df, n_topics=5)
if "error" in topics:
    print(f"  BERTopic skipped: {topics['error']}")
else:
    print(f"  Found {len(topics)} topics in negative reviews")
    for tid, info in topics.items():
        print(f"  Topic {tid} ({info['count']} reviews): {', '.join(info['top_words'][:5])}")

In [ ]:
# 5. Save all results
all_results = {
    'tfidf_baseline': tfidf_metrics,
    'distilbert':     bert_metrics if bert_metrics else {},
    'topics_negative': topics,
}
save_json(all_results, ARTEFACTS_DIR / 'nlp_results.json')
print(f"Saved → {ARTEFACTS_DIR / 'nlp_results.json'}")